# Model B — 7-Day Training + Temporal Holdout

Train on **Jul 13–19** and evaluate on the genuine future window **Jul 20–Aug 11**. Same 17 historical features, thresholds, RF parameters, and leakage logic.

In [2]:

import os
import json
import pickle
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from datetime import date
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    accuracy_score,
    confusion_matrix,
)

# Run from the project's notebooks/ directory.
# Project layout:
# ibm_quantum/
#   backup_calibration_history_20260811_1706.csv
#   models/
#   results/
#   notebooks/

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
CAL_CSV = os.path.join(PROJECT_ROOT, "backup_calibration_history_20260811_1706.csv")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

T1_THRESH = 100.0
T2_THRESH = 50.0
RE_THRESH = 0.05

HIST_FEATURES = [
    "hist_T1_mean", "hist_T1_std", "hist_T1_min", "hist_T1_max",
    "hist_T2_mean", "hist_T2_std", "hist_T2_min", "hist_T2_max",
    "hist_RE_mean", "hist_RE_std", "hist_RE_min", "hist_RE_max",
    "prev_T1", "prev_T2", "prev_RE",
    "hist_coherence_product", "hist_t2_t1_ratio"
]

RF_PARAMS = dict(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

print("Project root:", PROJECT_ROOT)
print("Calibration CSV:", CAL_CSV)

df = pd.read_csv(CAL_CSV)
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
for col in ["T1_us", "T2_us", "readout_error"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["qubit"] = pd.to_numeric(df["qubit"], errors="coerce").astype(int)
df = df.dropna(subset=["T1_us", "T2_us", "readout_error"])
df = df.sort_values(["backend", "qubit", "snapshot_date"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Date range: {df['snapshot_date'].min().date()} -> {df['snapshot_date'].max().date()}")
print(f"Unique calendar dates: {df['snapshot_date'].nunique()}")
print(f"Backends: {sorted(df['backend'].unique())}")
print(f"T1 mean: {df['T1_us'].mean():.1f} us")

assert df["T1_us"].mean() <= 10000, "T1 values look like a unit-conversion bug."

df["label"] = (
    (df["T1_us"] > T1_THRESH) &
    (df["T2_us"] > T2_THRESH) &
    (df["readout_error"] < RE_THRESH)
).astype(int)

# Same historical feature logic as Model A.
# Current-session values NEVER enter features.
df["snapshot_id"] = (
    df.groupby(["backend", "qubit"])["snapshot_date"]
      .transform(lambda x: pd.factorize(x)[0])
)

for col, alias in [("T1_us", "T1"), ("T2_us", "T2"), ("readout_error", "RE")]:
    g = df.groupby(["backend", "qubit"])[col]
    df[f"hist_{alias}_mean"] = g.transform(lambda x: x.shift(1).expanding().mean())
    df[f"hist_{alias}_std"]  = g.transform(lambda x: x.shift(1).expanding().std())
    df[f"hist_{alias}_min"]  = g.transform(lambda x: x.shift(1).expanding().min())
    df[f"hist_{alias}_max"]  = g.transform(lambda x: x.shift(1).expanding().max())
    df[f"prev_{alias}"]      = g.transform(lambda x: x.shift(1))

df["hist_coherence_product"] = df["hist_T1_mean"] * df["hist_T2_mean"]
df["hist_t2_t1_ratio"] = df["hist_T2_mean"] / (df["hist_T1_mean"] + 1e-9)

snap0 = df[df["snapshot_id"] == 0]
nan_rates = snap0[HIST_FEATURES].isnull().mean()
assert nan_rates.min() == 1.0, "LEAKAGE DETECTED: snapshot 0 has non-NaN historical features."
print("Leakage audit PASSED — snapshot 0 is 100% NaN")


# ============================================================
# MODEL B — TRAIN JUL 13-19, TEST JUL 20-AUG 11
# ============================================================

TRAIN_START, TRAIN_END = "2026-07-13", "2026-07-19"
TEST_START, TEST_END = "2026-07-20", "2026-08-11"

# Features were created over the complete timeline BEFORE the split.
# Therefore Jul 20 can use Jul 13-19 history, Jul 21 can use history
# through Jul 20, etc. No future values and no current-session values
# enter the feature vector.

train = df[
    (df["snapshot_date"] >= TRAIN_START) &
    (df["snapshot_date"] <= TRAIN_END)
].copy()

test = df[
    (df["snapshot_date"] >= TEST_START) &
    (df["snapshot_date"] <= TEST_END)
].copy()

train = train[train["snapshot_id"] >= 1].dropna(subset=HIST_FEATURES).copy()
test = test.dropna(subset=HIST_FEATURES).copy()

print("\n" + "=" * 65)
print("MODEL B — TRAINING")
print("=" * 65)
print("Training window:", TRAIN_START, "->", TRAIN_END)
print("Training dates:", train["snapshot_date"].nunique())
print("Training rows:", len(train))
print("Training viable rate:", round(train["label"].mean(), 4))
print("Backends:", sorted(train["backend"].unique()))

print("\nMODEL B — TEMPORAL TEST")
print("Test window:", TEST_START, "->", TEST_END)
print("Test dates:", test["snapshot_date"].nunique())
print("Test rows:", len(test))
print("Test viable rate:", round(test["label"].mean(), 4))
print("Backends:", sorted(test["backend"].unique()))

assert len(train) > 0
assert len(test) > 0
assert test["snapshot_date"].min() > train["snapshot_date"].max()

X_train = train[HIST_FEATURES]
y_train = train["label"]

model_b = RandomForestClassifier(**RF_PARAMS)
model_b.fit(X_train, y_train)

model_b_path = os.path.join(MODELS_DIR, "model_b_7day.pkl")
with open(model_b_path, "wb") as f:
    pickle.dump(model_b, f)

print("\nSaved Model B:", model_b_path)

# Genuine future temporal holdout.
X_test = test[HIST_FEATURES]
y_test = test["label"]

prob = model_b.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

auc = roc_auc_score(y_test, prob)
bal = balanced_accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
mcc = matthews_corrcoef(y_test, pred)
acc = accuracy_score(y_test, pred)

print("\n" + "=" * 65)
print("MODEL B — FINAL TEMPORAL HOLDOUT RESULT")
print("=" * 65)
print(f"AUC:                {auc:.4f}")
print(f"Balanced Accuracy:  {bal:.4f}")
print(f"F1:                 {f1:.4f}")
print(f"MCC:                {mcc:.4f}")
print(f"Accuracy:           {acc:.4f}")
print(f"Actual viable rate: {y_test.mean():.4f}")
print(f"Predicted viable:   {pred.mean():.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))

print("\nPer-backend results:")
backend_results = []
for backend in sorted(test["backend"].unique()):
    sub = test[test["backend"] == backend]
    if sub["label"].nunique() < 2:
        print(backend, "SKIPPED — only one class")
        continue
    bp = model_b.predict_proba(sub[HIST_FEATURES])[:, 1]
    bpred = (bp >= 0.5).astype(int)
    row = {
        "backend": backend,
        "rows": int(len(sub)),
        "AUC": float(roc_auc_score(sub["label"], bp)),
        "BalAcc": float(balanced_accuracy_score(sub["label"], bpred)),
        "F1": float(f1_score(sub["label"], bpred)),
        "MCC": float(matthews_corrcoef(sub["label"], bpred))
    }
    backend_results.append(row)
    print(row)

metadata = {
    "model_name": "Model_B",
    "model_file": "model_b_7day.pkl",
    "training_start": TRAIN_START,
    "training_end": TRAIN_END,
    "test_start": TEST_START,
    "test_end": TEST_END,
    "training_rows": int(len(train)),
    "test_rows": int(len(test)),
    "training_calendar_dates": int(train["snapshot_date"].nunique()),
    "test_calendar_dates": int(test["snapshot_date"].nunique()),
    "n_features": len(HIST_FEATURES),
    "features": HIST_FEATURES,
    "label_thresholds": {"T1_us": T1_THRESH, "T2_us": T2_THRESH, "RE": RE_THRESH},
    "feature_source": "prior_sessions_only_shift_expand",
    "leakage_status": "NONE_VERIFIED_BY_SNAPSHOT0_AUDIT",
    "rf_params": RF_PARAMS,
    "temporal_holdout": {
        "AUC": float(auc),
        "BalancedAccuracy": float(bal),
        "F1": float(f1),
        "MCC": float(mcc),
        "Accuracy": float(acc)
    },
    "backend_results": backend_results,
    "trained_on_date": str(date.today())
}

meta_path = os.path.join(RESULTS_DIR, "model_b_7day_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("\nMetadata saved:", meta_path)


Project root: D:\Downloads\uday projects\ibm_quantum
Calibration CSV: D:\Downloads\uday projects\ibm_quantum\backup_calibration_history_20260811_1706.csv
Loaded rows: 13,980
Date range: 2026-07-13 -> 2026-08-11
Unique calendar dates: 30
Backends: ['ibm_fez', 'ibm_kingston', 'ibm_marrakesh']
T1 mean: 177.4 us
Leakage audit PASSED — snapshot 0 is 100% NaN

MODEL B — TRAINING
Training window: 2026-07-13 -> 2026-07-19
Training dates: 5
Training rows: 2330
Training viable rate: 0.6292
Backends: ['ibm_fez', 'ibm_kingston', 'ibm_marrakesh']

MODEL B — TEMPORAL TEST
Test window: 2026-07-20 -> 2026-08-11
Test dates: 23
Test rows: 10718
Test viable rate: 0.6021
Backends: ['ibm_fez', 'ibm_kingston', 'ibm_marrakesh']

Saved Model B: D:\Downloads\uday projects\ibm_quantum\models\model_b_7day.pkl

MODEL B — FINAL TEMPORAL HOLDOUT RESULT
AUC:                0.8957
Balanced Accuracy:  0.8015
F1:                 0.8664
MCC:                0.6375
Accuracy:           0.8275
Actual viable rate: 0.6021
Pre